# بارگذاری داده‌ها

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

candidate_data_dirs = [
    Path("../data/dirty_data"),
    Path("data/dirty_data"),
]

data_dir = next((path for path in candidate_data_dirs if path.exists()), None)
if data_dir is None:
    raise FileNotFoundError("Dirty data folder was not found in ../data/dirty_data or data/dirty_data.")

csv_files = sorted(data_dir.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files were found under {data_dir}.")

tables = {}
for file_path in csv_files:
    tables[file_path.stem] = pd.read_csv(file_path)

if not tables:
    raise ValueError("No tables were loaded from the CSV files.")

project_root = data_dir.parent.parent.resolve()

print(f"Data path: {data_dir.resolve()}")
print(f"CSV count: {len(csv_files)}")
for table_name, df in tables.items():
    print(f"{table_name}: {df.shape}")


Data path: D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\data\dirty_data
CSV count: 13
Categories: (9, 4)
CustomerCustomerDemo: (0, 2)
CustomerDemographics: (0, 2)
Customers: (93, 11)
Employees: (10, 18)
EmployeeTerritories: (54, 2)
Order_Details: (2165, 5)
Orders: (835, 14)
Products: (79, 10)
Region: (5, 2)
Shippers: (4, 3)
Suppliers: (32, 12)
Territories: (59, 3)


# بررسی مقادیر گمشده

In [2]:
missing_rows = []

for table_name, df in tables.items():
    total_rows = len(df)

    for column in df.columns:
        missing_count = int(df[column].isna().sum())
        missing_percent = (missing_count / total_rows) * 100 if total_rows > 0 else 0

        missing_rows.append({
            "نام جدول": table_name,
            "نام ستون": column,
            "نوع داده": str(df[column].dtype),
            "تعداد ردیف": int(total_rows),
            "تعداد مقدار گمشده": missing_count,
            "درصد مقدار گمشده": float(missing_percent)
        })

missing_values_report = pd.DataFrame(missing_rows)

expected_columns = [
    "نام جدول",
    "نام ستون",
    "نوع داده",
    "تعداد ردیف",
    "تعداد مقدار گمشده",
    "درصد مقدار گمشده"
]

missing_values_report = missing_values_report.loc[:, expected_columns]

assert not missing_values_report.empty
assert missing_values_report.shape[1] == 6
assert "درصد مقدار گمشده" in missing_values_report.columns

print(missing_values_report.shape)
print(missing_values_report.columns.tolist())
display(missing_values_report.head())
display(missing_values_report)


(88, 6)
['نام جدول', 'نام ستون', 'نوع داده', 'تعداد ردیف', 'تعداد مقدار گمشده', 'درصد مقدار گمشده']


,نام جدول,نام ستون,نوع داده,تعداد ردیف,تعداد مقدار گمشده,درصد مقدار گمشده
0,Categories,CategoryID,int64,9,0,0.000000
1,Categories,CategoryName,object,9,0,0.000000
2,Categories,Description,object,9,4,44.444444
3,Categories,Picture,object,9,0,0.000000
4,CustomerCustomerDemo,CustomerID,object,0,0,0.000000


,نام جدول,نام ستون,نوع داده,تعداد ردیف,تعداد مقدار گمشده,درصد مقدار گمشده
0,Categories,CategoryID,int64,9,0,0.000000
1,Categories,CategoryName,object,9,0,0.000000
2,Categories,Description,object,9,4,44.444444
3,Categories,Picture,object,9,0,0.000000
4,CustomerCustomerDemo,CustomerID,object,0,0,0.000000
...,...,...,...,...,...,...
83,Suppliers,Fax,object,32,22,68.750000
84,Suppliers,HomePage,object,32,0,0.000000
85,Territories,TerritoryID,int64,59,0,0.000000
86,Territories,TerritoryDescription,object,59,0,0.000000


# ستون‌های دارای Missing بالا

In [3]:
high_missing_columns = missing_values_report[
    missing_values_report["درصد مقدار گمشده"] > 30
].copy()

if high_missing_columns.empty:
    print("No columns with Missing above 30 percent were found.")

display(high_missing_columns)


,نام جدول,نام ستون,نوع داده,تعداد ردیف,تعداد مقدار گمشده,درصد مقدار گمشده
2,Categories,Description,object,9,4,44.444444
14,Customers,Region,object,93,61,65.591398
18,Customers,Fax,object,93,31,33.333333
28,Employees,Region,object,10,5,50.000000
55,Orders,ShipRegion,object,835,511,61.197605
62,Products,QuantityPerUnit,object,79,26,32.911392
79,Suppliers,Region,object,32,22,68.750000
83,Suppliers,Fax,object,32,22,68.750000


# مقایسه Mean و Median

In [4]:
numeric_missing_rows = []

for table_name, df in tables.items():
    numeric_columns = df.select_dtypes(include=[np.number]).columns

    for column in numeric_columns:
        missing_count = int(df[column].isna().sum())
        if missing_count == 0:
            continue

        total_rows = len(df)
        missing_percent = (missing_count / total_rows) * 100 if total_rows > 0 else 0
        mean_value = df[column].mean()
        median_value = df[column].median()

        numeric_missing_rows.append({
            "نام جدول": table_name,
            "نام ستون": column,
            "تعداد Missing": missing_count,
            "درصد Missing": float(missing_percent),
            "Mean": mean_value,
            "Median": median_value,
            "اختلاف Mean و Median": abs(mean_value - median_value) if pd.notna(mean_value) and pd.notna(median_value) else np.nan,
        })

numeric_missing_summary = pd.DataFrame(numeric_missing_rows, columns=[
    "نام جدول",
    "نام ستون",
    "تعداد Missing",
    "درصد Missing",
    "Mean",
    "Median",
    "اختلاف Mean و Median",
])

display(numeric_missing_summary)


,نام جدول,نام ستون,تعداد Missing,درصد Missing,Mean,Median,اختلاف Mean و Median
0,Employees,ReportsTo,1,10.000000,4.444444,2.0,2.444444
1,Order_Details,UnitPrice,6,0.277136,28.619444,18.4,10.219444
2,Order_Details,Quantity,15,0.692841,25.002791,20.0,5.002791
3,Order_Details,Discount,19,0.877598,0.058616,0.0,0.058616


# بررسی داده‌های تکراری

In [5]:
duplicate_rows = []

for table_name, df in tables.items():
    total_rows = len(df)
    duplicate_count = int(df.duplicated().sum())
    duplicate_percent = (duplicate_count / total_rows) * 100 if total_rows > 0 else 0

    duplicate_rows.append({
        "نام جدول": table_name,
        "تعداد ردیف": int(total_rows),
        "تعداد ردیف تکراری": duplicate_count,
        "درصد ردیف تکراری": float(duplicate_percent),
    })

duplicate_rows_report = pd.DataFrame(duplicate_rows)
display(duplicate_rows_report)


,نام جدول,تعداد ردیف,تعداد ردیف تکراری,درصد ردیف تکراری
0,Categories,9,1,11.111111
1,CustomerCustomerDemo,0,0,0.000000
2,CustomerDemographics,0,0,0.000000
3,Customers,93,0,0.000000
4,Employees,10,0,0.000000
5,EmployeeTerritories,54,3,5.555556
6,Order_Details,2165,9,0.415704
7,Orders,835,0,0.000000
8,Products,79,1,1.265823
9,Region,5,0,0.000000


# بررسی ستون‌های متنی

In [6]:
text_whitespace_rows = []

for table_name, df in tables.items():
    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        values = df[column].dropna().astype(str)
        total_values = len(values)
        whitespace_count = int(values.ne(values.str.strip()).sum())
        whitespace_percent = (whitespace_count / total_values) * 100 if total_values > 0 else 0

        text_whitespace_rows.append({
            "نام جدول": table_name,
            "نام ستون": column,
            "تعداد مقدار دارای فاصله اضافی": whitespace_count,
            "درصد مقدار دارای فاصله اضافی": float(whitespace_percent),
        })

text_whitespace_report = pd.DataFrame(text_whitespace_rows, columns=[
    "نام جدول",
    "نام ستون",
    "تعداد مقدار دارای فاصله اضافی",
    "درصد مقدار دارای فاصله اضافی",
])

display(text_whitespace_report)


,نام جدول,نام ستون,تعداد مقدار دارای فاصله اضافی,درصد مقدار دارای فاصله اضافی
0,Categories,CategoryName,3,33.333333
1,Categories,Description,0,0.000000
2,Categories,Picture,0,0.000000
3,CustomerCustomerDemo,CustomerID,0,0.000000
4,CustomerCustomerDemo,CustomerTypeID,0,0.000000
5,CustomerDemographics,CustomerTypeID,0,0.000000
6,CustomerDemographics,CustomerDesc,0,0.000000
7,Customers,CustomerID,0,0.000000
8,Customers,CompanyName,20,21.505376
9,Customers,ContactName,15,16.129032


# بررسی Order Details

In [7]:
def normalize_table_name(name):
    return str(name).lower().replace(" ", "").replace("_", "").replace("-", "")

order_details_name = next(
    (table_name for table_name in tables if normalize_table_name(table_name) == "orderdetails"),
    None,
)

order_details_found = order_details_name is not None

if order_details_found:
    order_details_df = tables[order_details_name]
    total_rows = len(order_details_df)
    order_quality_rows = []

    for column, check_name, condition_type in [
        ("Quantity", "Quantity <= 0", "non_positive"),
        ("UnitPrice", "UnitPrice <= 0", "non_positive"),
        ("Quantity", "Quantity Missing", "missing"),
        ("UnitPrice", "UnitPrice Missing", "missing"),
    ]:
        if column not in order_details_df.columns:
            order_quality_rows.append({
                "بررسی": check_name,
                "تعداد": np.nan,
                "درصد": np.nan,
                "پیام": f"Column {column} was not found.",
            })
            continue

        if condition_type == "missing":
            issue_count = int(order_details_df[column].isna().sum())
        else:
            numeric_values = pd.to_numeric(order_details_df[column], errors="coerce")
            issue_count = int((numeric_values <= 0).sum())

        issue_percent = (issue_count / total_rows) * 100 if total_rows > 0 else 0
        order_quality_rows.append({
            "بررسی": check_name,
            "تعداد": issue_count,
            "درصد": float(issue_percent),
            "پیام": f"Checked in {order_details_name}.",
        })

    order_details_quality_summary = pd.DataFrame(order_quality_rows)
else:
    order_details_quality_summary = pd.DataFrame([{
        "پیام": "Order Details table was not found.",
    }])

display(order_details_quality_summary)


,بررسی,تعداد,درصد,پیام
0,Quantity <= 0,10,0.461894,Checked in Order_Details.
1,UnitPrice <= 0,6,0.277136,Checked in Order_Details.
2,Quantity Missing,15,0.692841,Checked in Order_Details.
3,UnitPrice Missing,6,0.277136,Checked in Order_Details.


# ذخیره گزارش‌ها

In [8]:
reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

report_outputs = {
    "missing_values_report.csv": missing_values_report,
    "high_missing_columns.csv": high_missing_columns,
    "numeric_missing_summary.csv": numeric_missing_summary,
    "duplicate_rows_report.csv": duplicate_rows_report,
    "text_whitespace_report.csv": text_whitespace_report,
    "order_details_quality_summary.csv": order_details_quality_summary,
}

for file_name, report_df in report_outputs.items():
    report_df.to_csv(reports_dir / file_name, index=False, encoding="utf-8-sig")

for file_name in report_outputs:
    print(reports_dir / file_name)


D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\missing_values_report.csv
D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\high_missing_columns.csv
D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\numeric_missing_summary.csv
D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\duplicate_rows_report.csv
D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\text_whitespace_report.csv
D:\دانشگاه\IT\ترم 4\Project\Project-IT\Phase-1\IT-14042\reports\order_details_quality_summary.csv


# بررسی پایانی

In [9]:
total_checked_columns = int(sum(len(df.columns) for df in tables.values()))
missing_column_count = int((missing_values_report["تعداد مقدار گمشده"] > 0).sum())
high_missing_count = int(len(high_missing_columns))
duplicate_table_count = int((duplicate_rows_report["تعداد ردیف تکراری"] > 0).sum())
order_details_status = "found" if order_details_found else "not found"

print(f"Loaded tables: {len(tables)}")
print(f"Checked columns: {total_checked_columns}")
print(f"Columns with Missing: {missing_column_count}")
print(f"Columns with Missing above 30 percent: {high_missing_count}")
print(f"Tables with duplicate rows: {duplicate_table_count}")
print(f"Order Details status: {order_details_status}")
print("Original dirty data files were not changed.")


Loaded tables: 13
Checked columns: 88
Columns with Missing: 23
Columns with Missing above 30 percent: 8
Tables with duplicate rows: 5
Order Details status: found
Original dirty data files were not changed.
